In [1]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery

from google.cloud import bigquery
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

# ── 1. Pull data ──────────────────────────────────────────────────────────────
client = bigquery.Client(project="cbb6790-final-project")
query = """
SELECT *
FROM `cbb6790-final-project.analysis.CBB5790_FinalProject`
"""
df = client.query(query).to_dataframe()

# ── 2. Create binary target ───────────────────────────────────────────────────
df["long_stay"] = (df["icu_length_of_stay"] >= 7).astype(int)

# ── 3. Define features ────────────────────────────────────────────────────────
df["gender_bin"] = (df["gender"] == "M").astype(int)

FEATURES = [
    "anchor_age",
    "creatinine_min", "creatinine_max",
    "bun_min", "bun_max",
    "potassium_min", "potassium_max",
    "bicarbonate_min", "bicarbonate_max",
    "sodium_min", "sodium_max",
    "mbp_min", "mbp_mean", "mbp_max",
    "heart_rate_min", "heart_rate_max",
    "urineoutput_24hr",
    "kdigo_stage",
    "gender_bin",
]
TARGET = "long_stay"

# ── 4. Stratified 80/10/10 split WITHIN each care unit ───────────────────────
train_dfs, tune_dfs, test_dfs = [], [], []
skipped_units = []

for unit in df["first_careunit"].dropna().unique():
    unit_df = df[df["first_careunit"] == unit].copy()

    if len(unit_df) < 30 or unit_df[TARGET].nunique() < 2:
        skipped_units.append((unit, len(unit_df), "too small or no outcome variation"))
        continue

    try:
        unit_trainval, unit_test = train_test_split(
            unit_df, test_size=0.10, random_state=42, stratify=unit_df[TARGET]
        )
        unit_train, unit_tune = train_test_split(
            unit_trainval, test_size=0.1111, random_state=42, stratify=unit_trainval[TARGET]
        )
        train_dfs.append(unit_train)
        tune_dfs.append(unit_tune)
        test_dfs.append(unit_test)

    except ValueError as e:
        skipped_units.append((unit, len(unit_df), str(e)))
        continue

df_train = pd.concat(train_dfs).reset_index(drop=True)
df_tune  = pd.concat(tune_dfs).reset_index(drop=True)
df_test  = pd.concat(test_dfs).reset_index(drop=True)

print("── Split sizes ──")
print(f"  Train: {len(df_train)} | Tune: {len(df_tune)} | Test: {len(df_test)}")
print(f"  Total accounted for: {len(df_train) + len(df_tune) + len(df_test)} / {len(df)} original rows")

print("\n── Long stay prevalence ──")
for name, split in [("Train", df_train), ("Tune", df_tune), ("Test", df_test)]:
    print(f"  {name}: {split[TARGET].mean():.3f}")

print("\n── Rows per care unit across splits ──")
for unit in df["first_careunit"].dropna().unique():
    n_train = (df_train["first_careunit"] == unit).sum()
    n_tune  = (df_tune["first_careunit"]  == unit).sum()
    n_test  = (df_test["first_careunit"]  == unit).sum()
    print(f"  {unit}: train={n_train}, tune={n_tune}, test={n_test}")

if skipped_units:
    print("\n── Skipped units (excluded from federation) ──")
    for unit, n, reason in skipped_units:
        print(f"  {unit} (n={n}): {reason}")

# ── 5. Helper functions ───────────────────────────────────────────────────────
def train_local_lasso(local_df, features, target, C=0.1):
    """Train one LASSO client model on its local training slice."""
    X = local_df[features]
    y = local_df[target]

    imputer = SimpleImputer(strategy="median")
    X_imp   = imputer.fit_transform(X)

    scaler   = StandardScaler()
    X_scaled = scaler.fit_transform(X_imp)

    model = LogisticRegression(
        penalty="l1",
        solver="liblinear",
        C=C,
        max_iter=1000,
        random_state=42,
    )
    model.fit(X_scaled, y)

    return {
        "coef":      model.coef_[0],
        "intercept": model.intercept_[0],
        "n_samples": len(local_df),
        "scaler":    scaler,
        "imputer":   imputer,
        "unit":      None,
    }


def federated_aggregate(client_results):
    """Weighted average of coefficients across clients (FedAvg)."""
    total_n   = sum(r["n_samples"] for r in client_results)
    agg_coef  = np.zeros_like(client_results[0]["coef"])
    agg_intercept = 0.0

    for r in client_results:
        weight         = r["n_samples"] / total_n
        agg_coef      += weight * r["coef"]
        agg_intercept += weight * r["intercept"]

    return agg_coef, agg_intercept


def score_global_model(coef, intercept, eval_df, features, target, ref):
    """Apply global coefficients to an evaluation set and return AUC."""
    X = eval_df[features]
    y = eval_df[target]

    X_imp    = ref["imputer"].transform(X)
    X_scaled = ref["scaler"].transform(X_imp)

    log_odds = X_scaled @ coef + intercept
    probs    = 1 / (1 + np.exp(-log_odds))

    return roc_auc_score(y, probs)


def fit_global_preprocessor(train_df, features):
    """Fit a single imputer+scaler on the full training set for fair evaluation."""
    imputer = SimpleImputer(strategy="median")
    scaler  = StandardScaler()
    X_imp   = imputer.fit_transform(train_df[features])
    scaler.fit(X_imp)
    return {"imputer": imputer, "scaler": scaler}

# ── 6. Tune C on validation set ───────────────────────────────────────────────
C_candidates = [0.001, 0.01, 0.1, 0.5, 1.0]
units = df_train["first_careunit"].dropna().unique()
global_ref = fit_global_preprocessor(df_train, FEATURES)

print("\n── Tuning C on validation set ──")
tune_results = {}

for C in C_candidates:
    client_results = []

    for unit in units:
        unit_df = df_train[df_train["first_careunit"] == unit].copy()
        if len(unit_df) < 30 or unit_df[TARGET].nunique() < 2:
            continue
        result = train_local_lasso(unit_df, FEATURES, TARGET, C=C)
        result["unit"] = unit
        client_results.append(result)

    if not client_results:
        continue

    global_coef, global_intercept = federated_aggregate(client_results)
    tune_auc = score_global_model(global_coef, global_intercept, df_tune, FEATURES, TARGET, global_ref)
    tune_results[C] = tune_auc
    print(f"  C={C:<6} → Tune AUC: {tune_auc:.4f}")

best_C = max(tune_results, key=tune_results.get)
print(f"\nBest C: {best_C} (Tune AUC: {tune_results[best_C]:.4f})")

# ── 7. Final federated training with best C ───────────────────────────────────
print(f"\n── Final federated training with C={best_C} ──")
final_client_results = []

for unit in units:
    unit_df = df_train[df_train["first_careunit"] == unit].copy()
    if len(unit_df) < 30 or unit_df[TARGET].nunique() < 2:
        continue
    result = train_local_lasso(unit_df, FEATURES, TARGET, C=best_C)
    result["unit"] = unit
    final_client_results.append(result)
    print(f"  ✓ {unit}: n={result['n_samples']}, "
          f"nonzero coefs={np.sum(result['coef'] != 0)}")

global_coef, global_intercept = federated_aggregate(final_client_results)

# ── 8. AUC Evaluation ─────────────────────────────────────────────────────────
print("\n── Global Model AUC ──")
for split_name, split_df in [("Train", df_train), ("Tune", df_tune), ("Test", df_test)]:
    auc = score_global_model(global_coef, global_intercept, split_df, FEATURES, TARGET, global_ref)
    print(f"  {split_name}: {auc:.4f}")

print("\n── Per-Unit Local Model Test AUC ──")
for r in final_client_results:
    try:
        X_imp    = r["imputer"].transform(df_test[FEATURES])
        X_scaled = r["scaler"].transform(X_imp)
        log_odds = X_scaled @ r["coef"] + r["intercept"]
        probs    = 1 / (1 + np.exp(-log_odds))
        auc      = roc_auc_score(df_test[TARGET], probs)
        print(f"  {r['unit']}: AUC = {auc:.4f}")
    except ValueError:
        print(f"  {r['unit']}: AUC could not be computed")

# ── 9. Global coefficient summary ─────────────────────────────────────────────
print("\n── Global Model Coefficients ──")
coef_df = pd.DataFrame({
    "feature":     FEATURES,
    "global_coef": global_coef,
}).sort_values("global_coef", key=abs, ascending=False)

print(coef_df.to_string(index=False))

── Split sizes ──
  Train: 24456 | Tune: 3061 | Test: 3061
  Total accounted for: 30578 / 30620 original rows

── Long stay prevalence ──
  Train: 0.193
  Tune: 0.194
  Test: 0.193

── Rows per care unit across splits ──
  Medical Intensive Care Unit (MICU): train=7614, tune=952, test=952
  Surgical Intensive Care Unit (SICU): train=2509, tune=314, test=314
  Medical/Surgical Intensive Care Unit (MICU/SICU): train=5136, tune=642, test=642
  Trauma SICU (TSICU): train=1864, tune=234, test=234
  Coronary Care Unit (CCU): train=3661, tune=458, test=458
  Cardiac Vascular Intensive Care Unit (CVICU): train=2740, tune=343, test=343
  Neuro Surgical Intensive Care Unit (Neuro SICU): train=310, tune=39, test=39
  Neuro Intermediate: train=416, tune=52, test=52
  Intensive Care Unit (ICU): train=0, tune=0, test=0
  PACU: train=38, tune=5, test=5
  Neuro Stepdown: train=77, tune=10, test=10
  Surgery/Vascular/Intermediate: train=91, tune=12, test=12
  Medicine: train=0, tune=0, test=0
  Surgery

In [2]:
# ── 10. Centralized Baseline (Pooled Data) ────────────────────────────────────
print("\n── Centralized Baseline Training ──")

central_imputer = SimpleImputer(strategy="median")
central_scaler = StandardScaler()

X_train_central_imp = central_imputer.fit_transform(df_train[FEATURES])
X_train_central_scaled = central_scaler.fit_transform(X_train_central_imp)
y_train_central = df_train[TARGET].values

central_model = LogisticRegression(
    penalty="l1",
    solver="liblinear",
    C=best_C,
    max_iter=1000,
    random_state=42,
)
central_model.fit(X_train_central_scaled, y_train_central)

print("\n── Centralized Model AUC ──")
for split_name, split_df in [("Train", df_train), ("Tune", df_tune), ("Test", df_test)]:
    X_eval_imp = central_imputer.transform(split_df[FEATURES])
    X_eval_scaled = central_scaler.transform(X_eval_imp)

    probs = central_model.predict_proba(X_eval_scaled)[:, 1]
    auc = roc_auc_score(split_df[TARGET], probs)
    print(f"  {split_name}: {auc:.4f}")

central_coef_df = pd.DataFrame({
    "feature": FEATURES,
    "central_coef": central_model.coef_[0]
}).sort_values("central_coef", key=abs, ascending=False)

print("\n── Centralized Model Coefficients ──")
print(central_coef_df.to_string(index=False))


── Centralized Baseline Training ──

── Centralized Model AUC ──
  Train: 0.8141
  Tune: 0.8041
  Test: 0.8153

── Centralized Model Coefficients ──
         feature  central_coef
     kdigo_stage      1.622724
urineoutput_24hr      0.235210
         bun_min      0.159964
      anchor_age     -0.153275
  creatinine_min     -0.128157
  creatinine_max     -0.127454
  heart_rate_max      0.117876
      gender_bin      0.110700
         bun_max     -0.103989
   potassium_min     -0.096753
 bicarbonate_min      0.094162
      sodium_max      0.091402
         mbp_min     -0.064535
         mbp_max      0.037979
   potassium_max      0.023892
        mbp_mean      0.018698
  heart_rate_min      0.005363
 bicarbonate_max      0.002037
      sodium_min      0.000000
